In [1]:
from IPython.display import display
import os
from pathlib import Path
import bacpipe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap

/workspaces/OsaAnurans3.11/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspaces/OsaAnurans3.11/.venv/lib/python3.11/site-packages/bacpipe/embedding_evaluation/visualization/dashboard.py:42: UserWarning: Using Panel interactively in VSCode notebooks requires the jupyter_bokeh package to be installed. You can install it with:

   pip install jupyter_bokeh

or:
    conda install jupyter_bokeh

and try again.
  pn.extension("plotly")


2026-10-15 20:18:39.691533: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-15 20:18:48.660432: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import importlib.resources as pkg_resources
os.chdir(pkg_resources.files("bacpipe"))
os.chdir('..')

MODEL_NAME = 'aves_especies'                  # name of the model to run. Supported models are in bacpipe.supported_models
MARSH_DIR = Path("data/audio/")
SWAMP_DIR = Path("data/swamp")
# Must match the results directory used when training the probe.
RESULTS_DIR = bacpipe.settings.main_results_dir

In [32]:
bacpipe.settings.device = 'cuda' 
bacpipe.config.models = ['aves_especies']

In [33]:
# Generate or load embeddings for annotated marsh recordings
marsh_loader = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,
    audio_dir=str(MARSH_DIR),
    check_if_already_processed=True,
)
marsh_embeds = marsh_loader.embeddings(return_type="array")

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

aves_especies checkpoint does not exists. Downloading the model from https://huggingface.co/datasets/vskode/bacpipe_models/blob/main/aves_especies/aves_especies.tar.xz




###### Generating embeddings using AVES_ESPECIES ######

finding audio files: 113it [00:00, 1226.24it/s]
Found 104 number of audio files.
Using device='cuda'
                                                                  

In [34]:
swamp_loader = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,
    audio_dir=str(SWAMP_DIR),
    check_if_already_processed=True,
)
swamp_embeds = swamp_loader.embeddings(return_type="array")

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

aves_especies checkpoint exists.




###### Generating embeddings using AVES_ESPECIES ######



Finding all generated embeddings: 277it [00:00, 17580.99it/s]
Found 277 embedding files.
finding audio files: 280it [00:00, 24986.28it/s]
Found 277 number of audio files.
The audio files are no longer where they used to be during the previous run. This might cause a problem.

### Embeddings already exist. Using embeddings in bacpipe_results/swamp/embeddings/2026-10-14_17-26___aves_especies-swamp ###


In [35]:
bacpipe.config.evaluation_task = ['probing']

In [36]:
# Align marsh annotations with the marsh embedding windows
marsh_gt = bacpipe.ground_truth_by_model(
    model=MODEL_NAME,
    audio_dir=str(MARSH_DIR),
    annotations_filename="annotations.csv",
    overwrite=True,
)

print("Embedding shape:", marsh_embeds.shape)
print("Ground truth shape:", marsh_gt.shape)
print("Ground truth type:", type(marsh_gt))


Multiple embeddings found for model aves_especies in bacpipe_results/swamp/embeddings. Using the most recent path.

No annotations found for audio_file='1Jsw1_20260917_193200.wav'. Continuing with next file.

No annotations found for audio_file='1Jsw1_20260917_193400.wav'. Continuing with next file.

No annotations found for audio_file='1Jsw1_20260917_195400.wav'. Continuing with next file.

No annotations found for audio_file='1Jsw1_20260917_201400.wav'. Continuing with next file.

No annotations found for audio_file='1Jsw1_20260917_203400.wav'. Continuing with next file.

No annotations found for audio_file='1Jsw1_20260917_205400.wav'. Continuing with next file.

No annotations found for audio_file='1Jsw1_20260917_211400.wav'. Continuing with next file.

No annotations found for audio_file='1Jsw1_20260917_213400.wav'. Continuing with next file.

No annotations found for audio_file='1Jsw1_20260917_215400.wav'. Continuing with next file.

No annotations found for audio_file='1Jsw1_202

Embedding shape: (6240, 768)
Ground truth shape: (0, 9)
Ground truth type: <class 'pandas.core.frame.DataFrame'>


In [37]:
# Train and evaluate the linear probe on marsh
probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name=MODEL_NAME,
    audio_dir=str(MARSH_DIR),
    ground_truth=marsh_gt,
    embeds=marsh_embeds,
    name="linear",
    overwrite=True,
)

print(metrics)
print(label2idx)


Found file: bacpipe_results/audio/evaluations/aves_especies/labels/probing_dataframe.csv. Loading dataframe probing dataframe. If you would like to automatically create a new probing dataframe. Please delete the existing one.


Multiple embeddings found for model aves_especies in bacpipe_results/audio/embeddings. Using the most recent path.

getting time of day: 100%|██████████| 104/104 [00:00<00:00, 367487.46it/s]
getting time per embeddings: 104it [00:00, 8009.25it/s]
getting day of year: 100%|██████████| 104/104 [00:00<00:00, 487382.81it/s]
getting continuous timestamps: 104it [00:00, 4948.19it/s]
getting parent directory: 104it [00:00, 22055.19it/s]
getting audio file names: 104it [00:00, 19559.13it/s]
Building metadata labels: 100%|██████████| 6/6 [00:00<00:00, 66.77it/s]


PermissionError: [Errno 13] Permission denied: 'bacpipe_results/audio/evaluations/aves_especies/labels/metadata_labels.csv'

In [ ]:
# Load the trained probe and its species-to-index mapping
probe, label2index = bacpipe.prepare_probe_inference(
    model=MODEL_NAME,
    audio_dir=str(MARSH_DIR),
    main_results_dir=RESULTS_DIR,
)

# Get class probabilities for each swamp embedding
probabilities = bacpipe.run_probe_inference(
    model=MODEL_NAME,
    linear_probe=probe,
    embeds=swamp_embeds,
    audio_dir=str(MARSH_DIR),
    main_results_dir=RESULTS_DIR,
    return_binary_presence=False,
)

# Bacpipe's class mapping maps species names to probability columns.
index_to_label = {
    index: label for label, index in label2index.items()
}

swamp_species = np.array([
    index_to_label[i]
    for i in probabilities.argmax(axis=1)
])

swamp_confidence = probabilities.max(axis=1)

if len(swamp_embeds) != len(swamp_species):
    raise ValueError("Swamp embeddings and predictions are misaligned.")

In [ ]:
# ---------------------------------------------------------------------
# 4. Fit UMAP on marsh, then project swamp into the same space
# ---------------------------------------------------------------------

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42,
)

marsh_2d = reducer.fit_transform(marsh_embeds)
swamp_2d = reducer.transform(swamp_embeds)

/workspaces/OsaAnurans3.11/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
marsh_2d

array([[22.266031, 11.291568],
       [22.14485 , 11.409028],
       [22.177748, 11.376518],
       ...,
       [16.83016 ,  9.164522],
       [16.734608,  9.275734],
       [16.815853,  9.146224]], dtype=float32)

In [ ]:
import numpy as np

species_cols = ["LEPFRA", "ELAPER", "ENGPUS", "DENEBR", "DENMIC"]

# Ground-truth species for marsh.
# This chooses the first present species if multiple are present.
marsh_species = marsh_gt[species_cols].idxmax(axis=1).to_numpy()

# Mark intervals with no species label as unknown.
marsh_species[marsh_gt[species_cols].sum(axis=1).to_numpy() == 0] = "Unknown"

# Predicted species for swamp.
index_to_label = {index: label for label, index in label2index.items()}
swamp_species = np.array([
    index_to_label[i] for i in probabilities.argmax(axis=1)
])

# Confidence of the top predicted species.
swamp_confidence = probabilities.max(axis=1)

assert len(marsh_embeds) == len(marsh_species)
assert len(swamp_embeds) == len(swamp_species)

In [ ]:
# ---------------------------------------------------------------------
# 5. Plot both datasets together
# ---------------------------------------------------------------------

plot_df = pd.concat(
    [
        pd.DataFrame({
            "UMAP 1": marsh_2d[:, 0],
            "UMAP 2": marsh_2d[:, 1],
            "dataset": "Marsh (known species)",
            "species": marsh_species,
            "confidence": np.nan,
        }),
        pd.DataFrame({
            "UMAP 1": swamp_2d[:, 0],
            "UMAP 2": swamp_2d[:, 1],
            "dataset": "Swamp (predicted species)",
            "species": swamp_species,
            "confidence": swamp_confidence,
        }),
    ],
    ignore_index=True,
)

# Save predictions for later inspection.
plot_df.to_csv(RESULTS_DIR + "marsh_swamp_umap_points.csv", index=False)

fig, ax = plt.subplots(figsize=(13, 9))

species_names = sorted(plot_df["species"].unique())
palette = plt.get_cmap("tab20", max(len(species_names), 1))
species_colors = {
    name: palette(i)
    for i, name in enumerate(species_names)
}

# Plot marsh first, then swamp on top so predictions remain visible.
for species in species_names:
    marsh_mask = (
        (plot_df["dataset"] == "Marsh (known species)")
        & (plot_df["species"] == species)
    )
    swamp_mask = (
        (plot_df["dataset"] == "Swamp (predicted species)")
        & (plot_df["species"] == species)
    )

    if marsh_mask.any():
        ax.scatter(
            plot_df.loc[marsh_mask, "UMAP 1"],
            plot_df.loc[marsh_mask, "UMAP 2"],
            color=species_colors[species],
            marker="o",
            s=18,
            alpha=0.45,
        )

    if swamp_mask.any():
        ax.scatter(
            plot_df.loc[swamp_mask, "UMAP 1"],
            plot_df.loc[swamp_mask, "UMAP 2"],
            color=species_colors[species],
            marker="^",
            s=35,
            alpha=0.9,
            edgecolors="black",
            linewidths=0.25,
        )

# Build a legend with species colours.
from matplotlib.lines import Line2D

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        color=species_colors[name],
        label=name,
        markersize=7,
    )
    for name in species_names
]
legend_handles.extend([
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        color="grey",
        label="Marsh: known label",
        markersize=7,
    ),
    Line2D(
        [0], [0],
        marker="^",
        linestyle="",
        color="grey",
        markeredgecolor="black",
        label="Swamp: predicted label",
        markersize=7,
    ),
])

ax.set_title("Marsh vs Swamp — shared UMAP embedding space")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.legend(
    handles=legend_handles,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize="small",
)
fig.tight_layout()
fig.savefig(RESULTS_DIR + "marsh_swamp_umap.png", dpi=200, bbox_inches="tight")
plt.show()

/tmp/ipykernel_29304/1741329761.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  plot_df = pd.concat(
